# Telecom Customer Churn - Preparacao de Dados para MLP

Este notebook foi refatorado para manter apenas a preparacao dos dados necessaria para uma MLP. As etapas de limpeza existentes foram reaproveitadas e organizadas em um fluxo reprodutivel, sem criar ou treinar modelos.

## 1. Imports e configuracao

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.30

DATA_PATH = Path("WA_Fn-UseC_-Telco-Customer-Churn.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("WA_Fn-UseC_-Telco-Customer-Churn_original.csv")

## 2. Carga e diagnostico inicial

In [ ]:
df_raw = pd.read_csv(DATA_PATH)

print('Arquivo:', DATA_PATH)
print('Linhas:', df_raw.shape[0], 'Colunas:', df_raw.shape[1])
print('Valores nulos detectados pelo pandas:', df_raw.isna().sum().sum())
print('Espacos em branco em TotalCharges:', df_raw['TotalCharges'].astype(str).str.strip().eq('').sum())

df_raw.head()

In [ ]:
df_raw['Churn'].value_counts(normalize=True).rename('proportion')

## 3. Limpeza reaproveitada do notebook original

Etapas mantidas: tratamento dos espacos em branco de `TotalCharges`, conversao para numerico, remocao de `customerID` e remocao de categorias redundantes geradas por servicos indisponiveis. O notebook original nao tinha uma etapa explicita de tratamento de outliers, entao nenhuma remocao de outliers foi adicionada aqui.

In [ ]:
df_clean = df_raw.copy()

# Reaproveita a regra original: os 11 espacos em branco de TotalCharges viram zero.
df_clean['TotalCharges'] = df_clean['TotalCharges'].replace({' ': '0'}).astype('float64')

# Identificador sem poder preditivo e sem utilidade para a MLP.
df_clean = df_clean.drop(columns=['customerID'])

# Mantem uma copia limpa ainda categorica para auditoria/EDA futura.
df_clean_categorical = df_clean.copy()

print('Linhas:', df_clean.shape[0], 'Colunas:', df_clean.shape[1])
print('Valores nulos apos limpeza:', df_clean.isna().sum().sum())
df_clean.head()

## 4. Transformacao de variaveis categoricas

A transformacao categorica segue a preparacao ja existente: variaveis binarias sao convertidas para 0/1 e variaveis com mais categorias recebem one-hot encoding. Depois disso, sao removidas as colunas redundantes que o notebook original ja eliminava.

In [ ]:
df_mlp = df_clean.copy()

# Mapeamento explicito do alvo para evitar dependencia da ordem dos dados.
df_mlp['Churn'] = df_mlp['Churn'].map({'No': 0, 'Yes': 1})

categorical_cols = [
    col for col in df_mlp.columns
    if col != 'Churn' and (df_mlp[col].dtype == 'object' or col == 'SeniorCitizen')
]

category_mappings = {'Churn': {'No': 0, 'Yes': 1}}
for col in categorical_cols:
    if df_mlp[col].nunique() == 2:
        unique_values = sorted(df_mlp[col].dropna().unique().tolist(), key=str)
        mapping = {category: code for code, category in enumerate(unique_values)}
        df_mlp[col] = df_mlp[col].map(mapping)
        category_mappings[col] = {str(category): int(code) for category, code in mapping.items()}
    else:
        df_mlp = pd.get_dummies(df_mlp, columns=[col], dtype=int)

redundant_columns = df_mlp.columns[df_mlp.columns.str.contains('No internet service')].tolist()
redundant_columns += [
    'MultipleLines_No phone service',
    'MultipleLines_No',
    'OnlineSecurity_No',
    'OnlineBackup_No',
    'DeviceProtection_No',
    'TechSupport_No',
    'StreamingTV_No',
    'StreamingMovies_No',
]
redundant_columns = [col for col in redundant_columns if col in df_mlp.columns]

df_mlp = df_mlp.drop(columns=redundant_columns)

print('Mapeamentos binarios:', category_mappings)
print('Colunas removidas por redundancia:', redundant_columns)
print('Shape final antes do split:', df_mlp.shape)
df_mlp.head()

## 5. Separacao entre features e alvo

In [ ]:
X_mlp = df_mlp.drop(columns=['Churn'])
y_mlp = df_mlp['Churn']

print('Features:', X_mlp.shape)
print('Target:', y_mlp.shape)
print('Distribuicao do target:')
print(y_mlp.value_counts(normalize=True).sort_index())

## 6. Split estratificado e escalonamento para MLP

MLPs sao sensiveis a escala. Por isso, o `StandardScaler` e ajustado somente no treino e aplicado no treino e teste. Nenhum modelo e criado nesta etapa.

In [ ]:
X_train_mlp, X_test_mlp, y_train_mlp, y_test_mlp = train_test_split(
    X_mlp,
    y_mlp,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_mlp,
)

scaler_mlp = StandardScaler()
X_train_mlp_scaled = pd.DataFrame(
    scaler_mlp.fit_transform(X_train_mlp),
    columns=X_train_mlp.columns,
    index=X_train_mlp.index,
)
X_test_mlp_scaled = pd.DataFrame(
    scaler_mlp.transform(X_test_mlp),
    columns=X_test_mlp.columns,
    index=X_test_mlp.index,
)

print('Treino:', X_train_mlp_scaled.shape, y_train_mlp.shape)
print('Teste:', X_test_mlp_scaled.shape, y_test_mlp.shape)
print('Target treino:')
print(y_train_mlp.value_counts(normalize=True).sort_index())
print('Target teste:')
print(y_test_mlp.value_counts(normalize=True).sort_index())

## 7. Objetos prontos para a futura MLP

Use `X_train_mlp_scaled`, `X_test_mlp_scaled`, `y_train_mlp` e `y_test_mlp` na etapa de modelagem. O objeto `scaler_mlp` deve ser reaproveitado para transformar novos dados com as mesmas features.

In [ ]:
mlp_preparation_artifacts = {
    'X_train': X_train_mlp_scaled,
    'X_test': X_test_mlp_scaled,
    'y_train': y_train_mlp,
    'y_test': y_test_mlp,
    'scaler': scaler_mlp,
    'feature_names': X_train_mlp_scaled.columns.tolist(),
    'category_mappings': category_mappings,
    'removed_redundant_columns': redundant_columns,
}

mlp_preparation_artifacts.keys()

## 8. MLPClassifier

A arquitetura fica concentrada em `MLP_HIDDEN_LAYER_SIZES`. Para testar outra rede, altere apenas essa tupla, por exemplo `(64,)`, `(64, 32)` ou `(128, 64, 32)`.

In [ ]:
from sklearn.neural_network import MLPClassifier

MLP_HIDDEN_LAYER_SIZES = (64, 32)
MLP_MAX_ITER = 500
MLP_LEARNING_RATE_INIT = 0.001

mlp_classifier = MLPClassifier(
    hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
    activation='relu',
    solver='adam',
    learning_rate_init=MLP_LEARNING_RATE_INIT,
    max_iter=MLP_MAX_ITER,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=RANDOM_STATE,
)

mlp_classifier.fit(X_train_mlp_scaled, y_train_mlp)

print('Arquitetura:', mlp_classifier.hidden_layer_sizes)
print('Iteracoes realizadas:', mlp_classifier.n_iter_)
print('Loss final:', mlp_classifier.loss_)

## 9. Artefatos da MLP

O objeto `mlp_model_artifacts` agrupa o modelo treinado e os dados preparados usados no treinamento. As metricas de avaliacao podem ser adicionadas em uma etapa separada.

In [ ]:
mlp_model_artifacts = {
    **mlp_preparation_artifacts,
    'model': mlp_classifier,
    'hidden_layer_sizes': MLP_HIDDEN_LAYER_SIZES,
    'max_iter': MLP_MAX_ITER,
    'learning_rate_init': MLP_LEARNING_RATE_INIT,
    'random_state': RANDOM_STATE,
}

mlp_model_artifacts.keys()